# Immune integration: annotation coverage diagnostic

Two read-only checks used to verify the TEA-seq well→suffix pairing and per-dataset annotation coverage after the loader fixes in `data_loading_utils.py` (2026-04-17).

- **Jaccard check**: confirms the hardcoded TEA-seq `well_to_suffix` mapping against barcode overlap between cellranger H5s and Figure4 `bc_seq`.
- **Coverage breakdown**: per-dataset fraction of cells with a non-NaN `harmonized_annotation`, with the failure mode classified (source-data gap vs. per-cell QC dropout vs. no annotation source).

Re-run this notebook if the source annotation CSV changes or after any change to the annotation-join logic in `data_loading_utils.py`.

In [ ]:
import os

import pandas as pd
import scanpy as sc

TEA_SEQ_BASE = "/nfs/team283/vk7/sanger_projects/large_data/tea_seq_pbmc"
ANNOTATION_CSV = os.path.join(TEA_SEQ_BASE, "supplementary_data/Figure4_SourceData2_TypeLabelsUMAP.csv")
SAMPLE_MAPPING = os.path.join(TEA_SEQ_BASE, "sample_mapping.csv")
ADATA_RNA = "/nfs/team205/vk7/sanger_projects/my_packages/regularizedvi/results/immune_integration/adata_rna.h5ad"

## 1. TEA-seq well → suffix Jaccard matrix

For each GSM in `sample_mapping.csv`, load the cellranger H5 barcode set. For each Figure4 Seurat-merge suffix (3/4/5/6), load the per-suffix `bc_seq` set. Jaccard = |A∩B|/|A∪B|; containment = |A∩B|/|B|.

A correct well→suffix mapping is the row-wise argmax with Jaccard > ~0.5 and containment > 0.99.

In [ ]:
ann = pd.read_csv(ANNOTATION_CSV, usecols=["barcode"])
ann["bc_seq"] = ann["barcode"].str.rsplit("-", n=1).str[0]
ann["suffix"] = ann["barcode"].str.rsplit("-", n=1).str[1]
suffix_sets = {str(s): set(sub["bc_seq"].unique()) for s, sub in ann.groupby("suffix")}

sm = pd.read_csv(SAMPLE_MAPPING)
gsm_sets: dict[str, set[str]] = {}
for _, row in sm.iterrows():
    sid = row["sample_id"]
    h5 = row["rna_h5_path"]
    if not os.path.exists(h5):
        continue
    ad = sc.read_10x_h5(h5)
    gsm_sets[sid] = {bc.rsplit("-", 1)[0] for bc in ad.obs_names}
    print(f"  {sid}: {len(gsm_sets[sid])} bc_seq")


def jacc(a, b):
    """Jaccard similarity |A∩B| / |A∪B|."""
    return len(a & b) / len(a | b) if a and b else 0.0


def contain(a, b):
    """Containment |A∩B| / |B| — fraction of suffix set found in GSM."""
    return len(a & b) / len(b) if b else 0.0


suffixes = sorted(suffix_sets)
gsms = sorted(gsm_sets)
jmat = pd.DataFrame(
    [[jacc(gsm_sets[g], suffix_sets[s]) for s in suffixes] for g in gsms],
    index=gsms,
    columns=suffixes,
)
cmat = pd.DataFrame(
    [[contain(gsm_sets[g], suffix_sets[s]) for s in suffixes] for g in gsms],
    index=gsms,
    columns=suffixes,
)
print("\nJaccard (rows=GSM, cols=suffix):")
print(jmat.round(3))
print("\nContainment (|GSM ∩ suffix| / |suffix|):")
print(cmat.round(3))
print("\nArgmax suffix per GSM (by Jaccard):")
for g in gsms:
    print(f"  {g}: suffix '{jmat.loc[g].idxmax()}' (J={jmat.loc[g].max():.3f})")

## 2. Per-dataset annotation coverage (reads the final combined `adata_rna.h5ad`)

Reports for each `dataset` in the combined RNA AnnData:
- total cells
- cells with `harmonized_annotation` non-NaN
- breakdown by `batch` so per-sample zero-coverage (source-data gap) is distinguishable from per-cell partial coverage (post-QC dropout).

In [ ]:
adata = sc.read_h5ad(ADATA_RNA)
print(f"Combined adata: {adata.n_obs} cells x {adata.n_vars} genes\n")

summary_rows = []
for ds in sorted(adata.obs["dataset"].unique()):
    sub = adata.obs[adata.obs["dataset"] == ds]
    n = len(sub)
    n_ann = sub["harmonized_annotation"].notna().sum()
    n_orig = sub["original_annotation"].notna().sum()
    summary_rows.append(
        {
            "dataset": ds,
            "n_cells": n,
            "n_original_annot": int(n_orig),
            "n_harmonized": int(n_ann),
            "pct_harmonized": 100.0 * n_ann / n,
        }
    )
summary = pd.DataFrame(summary_rows).sort_values("dataset")
print(summary.to_string(index=False))

In [ ]:
# Per-batch breakdown for datasets that expose partial coverage
for ds in sorted(adata.obs["dataset"].unique()):
    sub = adata.obs[adata.obs["dataset"] == ds]
    total = sub["harmonized_annotation"].notna().sum()
    if total == 0 or total == len(sub):
        continue  # fully unannotated or fully annotated
    print(f"\n=== {ds} (partial coverage) ===")
    per_batch = (
        sub.groupby("batch", observed=True)
        .agg(
            n_cells=("harmonized_annotation", "size"),
            n_annotated=("harmonized_annotation", lambda x: x.notna().sum()),
        )
        .assign(pct=lambda df: 100 * df["n_annotated"] / df["n_cells"])
        .sort_values("pct")
    )
    print(per_batch.to_string())